# Face Occlusion — Kaggle V2 (Target: < 0.00092)
**3-model ensemble** on P100 16GB: ConvNeXt-Base-CLIP + EVA-02-Base + EfficientNetV2-S

Key features: EMA, differential LR, gender-balanced sampling, 6-TTA, OOF-tuned blend

**Resume-safe**: saves progress after each epoch/fold. Re-run to continue where you left off.

In [ ]:
import subprocess, sys
def pipi(*args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])
try:
    import timm
    from packaging import version
    if version.parse(timm.__version__) < version.parse('1.0.19'):
        pipi('-U', 'timm')
except Exception:
    pipi('-U', 'timm')

In [ ]:
import os, gc, math, random, warnings, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from torch.amp import GradScaler, autocast
    def amp_ctx(): return autocast('cuda')
    def make_scaler(): return GradScaler('cuda')
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
    def amp_ctx(): return autocast()
    def make_scaler(): return GradScaler()

import torchvision.transforms as T
import timm
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')
print('torch', torch.__version__, '| timm', timm.__version__)

In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')

# ============ CONFIGURATION ============
CFG = {
    'seed': 42,
    'image_dir': '/kaggle/input/datasets/ahmedfakhfahk/face-occlusion/face_occlusion_images/Crop_224_5fp_100K' if IS_KAGGLE else 'Crop_224_5fp_100K',
    'train_csv': '/kaggle/input/datasets/ahmedfakhfahk/face-occlusion/occlusion_datasets/train.csv' if IS_KAGGLE else 'occlusion_datasets/train.csv',
    'test_csv': '/kaggle/input/datasets/ahmedfakhfahk/face-occlusion/occlusion_datasets/test_students.csv' if IS_KAGGLE else 'occlusion_datasets/test_students.csv',
    'output_dir': '/kaggle/working' if IS_KAGGLE else './checkpoints_v2',
    'img_size': 224,
    'n_folds': 5,
    'train_folds': [0, 1, 2],  # 3-fold for Kaggle time limit (use 5 locally)
    'num_workers': 2 if IS_KAGGLE else 4,
    'balanced_gender_sampling': True,
    'use_ema': True,
    'ema_decay': 0.999,
    'use_tta': True,
    'grad_accum': 1,       # Gradient accumulation steps (effective batch = batch_size * accum)
    'patience': 6,
    'weight_decay': 1e-4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

# 3-model ensemble tuned for 8GB VRAM
MODEL_ZOO = [
    {
        'name': 'convnext_base.clip_laion2b_augreg_ft_in12k_in1k',
        'batch_size': 48,
        'lr': 8e-5,
        'epochs': 12,
        'backbone_lr_mult': 0.3,
    },
    {
        'name': 'eva02_base_patch14_224.mim_in22k',
        'batch_size': 40,
        'lr': 6e-5,
        'epochs': 12,
        'backbone_lr_mult': 0.2,
    },
    {
        'name': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'batch_size': 64,
        'lr': 1e-4,
        'epochs': 14,
        'backbone_lr_mult': 0.5,
    },
]

os.makedirs(CFG['output_dir'], exist_ok=True)

def seed_everything(seed):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_everything(CFG['seed'])

print(f"Device: {CFG['device']}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB')
print(f'Models: {len(MODEL_ZOO)} | Folds: {CFG["train_folds"]} | Grad accum: {CFG["grad_accum"]}')

In [ ]:
# ============ DATA & FOLDS ============
df_train = pd.read_csv(CFG['train_csv']).dropna().reset_index(drop=True)
df_test = pd.read_csv(CFG['test_csv']).dropna().reset_index(drop=True)
print(f'train: {len(df_train):,} | test: {len(df_test):,}')

df_train['occ_bin'] = pd.cut(df_train['FaceOcclusion'], bins=10, labels=False)
df_train['strat'] = df_train['gender'].astype(int).astype(str) + '_' + df_train['occ_bin'].astype(str)
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
df_train['fold'] = -1
for f, (_, vidx) in enumerate(skf.split(df_train, df_train['strat'])):
    df_train.loc[vidx, 'fold'] = f

print(df_train.groupby('gender')['FaceOcclusion'].describe().round(4))

In [ ]:
# ============ TRANSFORMS ============
def build_transforms(mean, std, img_size):
    norm = T.Normalize(mean=mean, std=std)
    train = T.Compose([
        T.Resize((img_size, img_size)),
        T.RandomHorizontalFlip(0.5),
        T.RandomAffine(degrees=15, translate=(0.06, 0.06), scale=(0.88, 1.12)),
        T.ColorJitter(0.25, 0.25, 0.2, 0.05),
        T.RandomGrayscale(p=0.05),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
        T.ToTensor(), norm,
        T.RandomErasing(p=0.3, scale=(0.02, 0.2)),
    ])
    val = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), norm])
    tta = [
        val,
        T.Compose([T.Resize((img_size, img_size)), T.RandomHorizontalFlip(1.0), T.ToTensor(), norm]),
        T.Compose([T.Resize((img_size, img_size)), T.ColorJitter(brightness=(1.1, 1.1)), T.ToTensor(), norm]),
        T.Compose([T.Resize((img_size, img_size)), T.ColorJitter(brightness=(0.9, 0.9)), T.ToTensor(), norm]),
        T.Compose([T.Resize((img_size, img_size)), T.ColorJitter(contrast=(1.1, 1.1)), T.ToTensor(), norm]),
        T.Compose([T.Resize((img_size, img_size)), T.RandomHorizontalFlip(1.0),
                   T.ColorJitter(brightness=(1.05, 1.05)), T.ToTensor(), norm]),
    ]
    return train, val, tta


class FaceDS(Dataset):
    def __init__(self, df, image_dir, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.dir = image_dir
        self.tf = transform
        self.is_test = is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(f"{self.dir}/{r['filename']}").convert('RGB')
        img = self.tf(img)
        if self.is_test:
            return img, r['filename']
        return img, torch.tensor(r['FaceOcclusion'], dtype=torch.float32), \
               torch.tensor(r['gender'], dtype=torch.float32)

In [ ]:
# ============ MODEL + EMA + LOSS ============
class OcclusionModel(nn.Module):
    def __init__(self, name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0)
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat),
            nn.Dropout(0.3),
            nn.Linear(feat, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )
        self.name = name
    def forward(self, x):
        return torch.sigmoid(self.head(self.backbone(x))).squeeze(-1)


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v)
    def state_dict(self):
        return self.shadow
    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


class WeightedMSE(nn.Module):
    def forward(self, pred, target):
        w = 1.0 / 30.0 + target
        return (w * (pred - target) ** 2).sum() / w.sum()


def weighted_err(pred, target):
    w = 1.0 / 30.0 + target
    return float(np.sum(w * (pred - target) ** 2) / np.sum(w))

def official_score(pred, target, gender):
    mf, mm = gender == 0.0, gender == 1.0
    ef = weighted_err(pred[mf], target[mf]) if mf.sum() else 0.0
    em = weighted_err(pred[mm], target[mm]) if mm.sum() else 0.0
    return (ef + em) / 2.0 + abs(ef - em), ef, em

print('Model + metric defined.')

In [ ]:
# ============ TRAINING UTILITIES ============
def make_param_groups(model, base_lr, backbone_mult, wd=1e-4):
    head_p, bb_p = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (head_p if n.startswith('head') else bb_p).append(p)
    return [
        {'params': bb_p, 'lr': base_lr * backbone_mult, 'weight_decay': wd},
        {'params': head_p, 'lr': base_lr, 'weight_decay': wd},
    ]


def gender_balanced_sampler(df):
    counts = df['gender'].value_counts().to_dict()
    w = df['gender'].map(lambda g: 1.0 / counts[g]).values
    return WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                 num_samples=len(df), replacement=True)


def mixup_data(x, y, alpha=0.2):
    """MixUp augmentation for regression - improves generalization."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    mixed_y = lam * y + (1 - lam) * y[idx]
    return mixed_x, mixed_y

In [ ]:
# ============ TRAIN ONE FOLD ============
CHECKPOINT_DIR = os.path.join(CFG['output_dir'], 'progress')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def get_progress_path(name, fold):
    """Path for saving/resuming training progress."""
    safe_name = name.replace('/', '_').replace('.', '_')
    return os.path.join(CHECKPOINT_DIR, f'progress_{safe_name}_f{fold}.pt')


def save_progress(name, fold, epoch, model, ema, optimizer, scheduler, scaler, best_score, best_oof, patience_counter):
    """Save full training state for resume."""
    state = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_score': best_score,
        'best_oof': best_oof,
        'patience_counter': patience_counter,
    }
    if ema is not None:
        state['ema_shadow'] = ema.shadow
    torch.save(state, get_progress_path(name, fold))


def load_progress(name, fold):
    """Load training state if exists."""
    path = get_progress_path(name, fold)
    if os.path.exists(path):
        return torch.load(path, map_location='cpu', weights_only=False)
    return None


def train_fold(mcfg, fold, df, mean, std):
    dev = CFG['device']
    name = mcfg['name']; img_size = CFG['img_size']
    accum = CFG['grad_accum']
    train_tf, val_tf, _ = build_transforms(mean, std, img_size)

    tr = df[df.fold != fold]; va = df[df.fold == fold]
    print(f"\n{'='*70}")
    print(f"{name} | fold {fold} | train {len(tr):,} val {len(va):,} "
          f"(F={int((va.gender==0).sum())}, M={int((va.gender==1).sum())})")
    print(f"{'='*70}")

    tr_ds = FaceDS(tr, CFG['image_dir'], train_tf)
    va_ds = FaceDS(va, CFG['image_dir'], val_tf)
    
    if CFG['balanced_gender_sampling']:
        tr_loader = DataLoader(tr_ds, batch_size=mcfg['batch_size'],
                               sampler=gender_balanced_sampler(tr),
                               num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    else:
        tr_loader = DataLoader(tr_ds, batch_size=mcfg['batch_size'], shuffle=True,
                               num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    va_loader = DataLoader(va_ds, batch_size=mcfg['batch_size'] * 2, shuffle=False,
                           num_workers=CFG['num_workers'], pin_memory=True)

    model = OcclusionModel(name, pretrained=True).to(dev)
    model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW(
        make_param_groups(model, mcfg['lr'], mcfg['backbone_lr_mult'], CFG['weight_decay']))
    
    steps_per_epoch = len(tr_loader) // accum if accum > 1 else len(tr_loader)
    total_steps = steps_per_epoch * mcfg['epochs']
    warmup_steps = steps_per_epoch * 1
    
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g['lr'] for g in opt.param_groups],
        total_steps=total_steps, pct_start=warmup_steps / total_steps,
        anneal_strategy='cos', div_factor=10, final_div_factor=100)
    
    scaler = make_scaler()
    crit = WeightedMSE()
    ema = EMA(model, decay=CFG['ema_decay']) if CFG['use_ema'] else None

    best = math.inf; best_oof = None; patience_counter = 0
    start_epoch = 0
    save_path = os.path.join(CFG['output_dir'], f'best_{name.replace("/", "_")}_f{fold}.pth')

    # ---- RESUME FROM CHECKPOINT ----
    ckpt = load_progress(name, fold)
    if ckpt is not None:
        start_epoch = ckpt['epoch'] + 1
        model.load_state_dict(ckpt['model_state'])
        opt.load_state_dict(ckpt['optimizer_state'])
        sched.load_state_dict(ckpt['scheduler_state'])
        scaler.load_state_dict(ckpt['scaler_state'])
        best = ckpt['best_score']
        best_oof = ckpt['best_oof']
        patience_counter = ckpt['patience_counter']
        if ema is not None and 'ema_shadow' in ckpt:
            ema.shadow = {k: v.to(dev) for k, v in ckpt['ema_shadow'].items()}
        print(f"  >> RESUMED from epoch {start_epoch} (best={best:.6f}, patience={patience_counter})")
        del ckpt; gc.collect()
    # ---- END RESUME ----

    @torch.no_grad()
    def evaluate(eval_model):
        eval_model.eval()
        P, Tt, G = [], [], []
        for x, y, g in va_loader:
            x = x.to(dev, memory_format=torch.channels_last)
            with amp_ctx():
                p = eval_model(x)
            P.append(p.float().cpu().numpy())
            Tt.append(y.numpy()); G.append(g.numpy())
        P, Tt, G = np.concatenate(P), np.concatenate(Tt), np.concatenate(G)
        return official_score(P, Tt, G) + (P,)

    for ep in range(start_epoch, mcfg['epochs']):
        model.train()
        run_loss = 0.0; n_steps = 0
        opt.zero_grad()
        pbar = tqdm(tr_loader, desc=f'ep {ep+1}/{mcfg["epochs"]}')
        
        for batch_idx, (x, y, g) in enumerate(pbar):
            x = x.to(dev, memory_format=torch.channels_last)
            y = y.to(dev)
            
            # MixUp 50% of the time
            if random.random() < 0.5:
                x, y = mixup_data(x, y, alpha=0.2)
            
            with amp_ctx():
                loss = crit(model(x), y) / accum
            
            scaler.scale(loss).backward()
            
            if accum <= 1 or (batch_idx + 1) % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
                sched.step()
                opt.zero_grad()
                if ema: ema.update(model)
                n_steps += 1
            
            run_loss += loss.item() * accum
            pbar.set_postfix(loss=f'{run_loss/(batch_idx+1):.5f}')

        # Evaluate EMA model
        if ema:
            tmp = OcclusionModel(name, pretrained=False).to(dev).to(memory_format=torch.channels_last)
            ema.copy_to(tmp)
            s, ef, em, oof = evaluate(tmp)
            del tmp
        else:
            s, ef, em, oof = evaluate(model)

        avg_loss = run_loss / len(tr_loader)
        print(f'  loss {avg_loss:.5f} | score {s:.6f} | err_F {ef:.6f} err_M {em:.6f} | gap {abs(ef-em):.6f}')

        if s < best:
            best, best_oof, patience_counter = s, oof, 0
            torch.save(ema.state_dict() if ema else model.state_dict(), save_path)
            print(f'  >> NEW BEST {best:.6f} saved')
        else:
            patience_counter += 1
            if patience_counter >= CFG['patience']:
                print(f'  >> Early stop @ ep {ep+1}')
                # Save final progress before breaking
                save_progress(name, fold, ep, model, ema, opt, sched, scaler, best, best_oof, patience_counter)
                break

        # Save checkpoint after every epoch for resume capability
        save_progress(name, fold, ep, model, ema, opt, sched, scaler, best, best_oof, patience_counter)

    # Training complete for this fold — remove progress checkpoint
    prog_path = get_progress_path(name, fold)
    if os.path.exists(prog_path):
        os.remove(prog_path)
        print(f'  >> Progress checkpoint removed (training complete)')

    del model, opt, sched, scaler, tr_loader, va_loader
    gc.collect(); torch.cuda.empty_cache()
    return {'score': best, 'oof': best_oof, 'val_idx': va.index.values,
            'ckpt': save_path, 'name': name}

In [ ]:
# ============ TRAIN ALL MODELS x FOLDS ============
# Resume-aware: if results already exist for a model+fold, skip it.

RESULTS_PATH = os.path.join(CFG['output_dir'], 'all_results.pt')

# Load previous results if resuming
if os.path.exists(RESULTS_PATH):
    saved = torch.load(RESULTS_PATH, map_location='cpu', weights_only=False)
    oof_pred = saved['oof_pred']
    oof_filled = saved['oof_filled']
    model_meta = saved['model_meta']
    print(f"RESUMED: loaded previous results from {RESULTS_PATH}")
    for name in model_meta:
        n_done = len(model_meta[name]['results'])
        print(f"  {name}: {n_done} folds done")
else:
    oof_pred = {m['name']: np.zeros(len(df_train), dtype=np.float64) for m in MODEL_ZOO}
    oof_filled = {m['name']: np.zeros(len(df_train), dtype=bool) for m in MODEL_ZOO}
    model_meta = {}

start = time.time()
for mcfg in MODEL_ZOO:
    name = mcfg['name']
    
    # Get model's native normalization
    probe = OcclusionModel(name, pretrained=False)
    dc = timm.data.resolve_model_data_config(probe.backbone)
    mean, std = dc['mean'], dc['std']
    del probe; gc.collect()
    
    # Initialize meta if new model
    if name not in model_meta:
        model_meta[name] = {'mean': mean, 'std': std, 'results': []}
    
    # Determine which folds are already done
    done_folds = set()
    for r in model_meta[name]['results']:
        # Find which fold this result corresponds to by checking val_idx
        for fold in CFG['train_folds']:
            va = df_train[df_train.fold == fold]
            if set(r['val_idx']).issubset(set(va.index.values)):
                done_folds.add(fold)
                break
    
    remaining_folds = [f for f in CFG['train_folds'] if f not in done_folds]
    
    if not remaining_folds:
        print(f"\n{'#'*70}")
        print(f"# {name} — ALL FOLDS COMPLETE, SKIPPING")
        print(f"{'#'*70}")
        continue
    
    print(f"\n{'#'*70}")
    print(f"# {name}")
    print(f"# mean={tuple(round(x,3) for x in mean)} std={tuple(round(x,3) for x in std)}")
    print(f"# batch={mcfg['batch_size']} | folds remaining: {remaining_folds}")
    print(f"{'#'*70}")
    
    for fold in remaining_folds:
        res = train_fold(mcfg, fold, df_train, mean, std)
        oof_pred[name][res['val_idx']] = res['oof']
        oof_filled[name][res['val_idx']] = True
        model_meta[name]['results'].append(res)
        print(f"  -> fold {fold}: {res['score']:.6f}")
        
        # Save results after each fold (crash-safe)
        torch.save({
            'oof_pred': oof_pred,
            'oof_filled': oof_filled,
            'model_meta': model_meta,
        }, RESULTS_PATH)
        print(f"  >> Results saved to {RESULTS_PATH}")
    
    scores = [r['score'] for r in model_meta[name]['results']]
    print(f"\n{name} mean CV: {np.mean(scores):.6f} (std: {np.std(scores):.6f})")

print(f"\n{'='*70}")
print(f'Total training time: {(time.time()-start)/3600:.2f} h')
print(f"{'='*70}")

In [ ]:
# ============ OOF SCORES PER MODEL ============
g = df_train['gender'].values
y = df_train['FaceOcclusion'].values

print('Individual model OOF scores:')
print('-' * 60)
for name in oof_pred:
    mask = oof_filled[name]
    s, ef, em = official_score(oof_pred[name][mask], y[mask], g[mask])
    print(f'{name[:45]:45s} | {s:.6f} | F:{ef:.6f} M:{em:.6f} gap:{abs(ef-em):.6f}')

In [ ]:
# ============ TUNE ENSEMBLE WEIGHTS ON OOF ============
names = list(oof_pred.keys())
common = np.ones(len(df_train), dtype=bool)
for n in names:
    common &= oof_filled[n]

print(f'Common OOF samples: {common.sum():,}')

if len(names) == 1:
    blend_w = {names[0]: 1.0}
    print('Single model — no blending needed.')
elif len(names) == 2:
    best_s, best_w = math.inf, None
    for a in np.linspace(0, 1, 51):
        mix = a * oof_pred[names[0]][common] + (1-a) * oof_pred[names[1]][common]
        s, _, _ = official_score(mix, y[common], g[common])
        if s < best_s:
            best_s, best_w = s, {names[0]: a, names[1]: 1-a}
    blend_w = best_w
else:
    # Grid search over simplex for 3 models
    best_s, best_w = math.inf, None
    grid = np.linspace(0, 1, 21)
    for a in grid:
        for b in grid:
            if a + b > 1: continue
            c = 1 - a - b
            mix = (a * oof_pred[names[0]][common] + 
                   b * oof_pred[names[1]][common] + 
                   c * oof_pred[names[2]][common])
            s, _, _ = official_score(mix, y[common], g[common])
            if s < best_s:
                best_s, best_w = s, {names[0]: a, names[1]: b, names[2]: c}
    blend_w = best_w
    # Fine-tune around best with finer grid
    center = list(blend_w.values())
    fine_best_s, fine_best_w = best_s, best_w
    for da in np.linspace(-0.1, 0.1, 21):
        for db in np.linspace(-0.1, 0.1, 21):
            a2 = center[0] + da; b2 = center[1] + db; c2 = 1 - a2 - b2
            if a2 < 0 or b2 < 0 or c2 < 0: continue
            mix = (a2 * oof_pred[names[0]][common] + 
                   b2 * oof_pred[names[1]][common] + 
                   c2 * oof_pred[names[2]][common])
            s, _, _ = official_score(mix, y[common], g[common])
            if s < fine_best_s:
                fine_best_s = s
                fine_best_w = {names[0]: a2, names[1]: b2, names[2]: c2}
    blend_w = fine_best_w
    best_s = fine_best_s

# Print final blend
print(f'\nOptimal blend weights:')
for n, w in blend_w.items():
    print(f'  {n[:50]:50s}: {w:.3f}')

mix = sum(blend_w[n] * oof_pred[n][common] for n in names)
s, ef, em = official_score(mix, y[common], g[common])
print(f'\nBlended OOF score: {s:.6f} | err_F {ef:.6f} err_M {em:.6f} | gap {abs(ef-em):.6f}')

In [ ]:
# ============ INFERENCE WITH TTA ============
@torch.no_grad()
def predict_test(name, ckpt, df, image_dir, mean, std, dev):
    model = OcclusionModel(name, pretrained=False).to(dev).to(memory_format=torch.channels_last)
    sd = torch.load(ckpt, map_location=dev, weights_only=True)
    model.load_state_dict(sd, strict=True)
    model.eval()
    
    _, val_tf, tta = build_transforms(mean, std, CFG['img_size'])
    transforms = tta if CFG['use_tta'] else [val_tf]
    
    acc = None
    for t_idx, tf in enumerate(transforms):
        ds = FaceDS(df, image_dir, tf, is_test=True)
        loader = DataLoader(ds, batch_size=64, shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True)
        chunks = []
        for x, _ in tqdm(loader, desc=f'TTA {t_idx+1}/{len(transforms)}', leave=False):
            x = x.to(dev, memory_format=torch.channels_last)
            with amp_ctx():
                chunks.append(model(x).float().cpu().numpy())
        p = np.concatenate(chunks)
        acc = p if acc is None else acc + p
    
    del model; gc.collect(); torch.cuda.empty_cache()
    return acc / len(transforms)


print('Generating test predictions...')
test_pred = {}
for name, meta in model_meta.items():
    fold_preds = []
    for res in meta['results']:
        fp = predict_test(res['name'], res['ckpt'], df_test,
                         CFG['image_dir'], meta['mean'], meta['std'], CFG['device'])
        fold_preds.append(fp)
    test_pred[name] = np.mean(fold_preds, axis=0)
    print(f'{name}: test preds ready (mean={test_pred[name].mean():.4f})')

# Blend
final = sum(blend_w[n] * test_pred[n] for n in test_pred)
final = np.clip(final, 0.0, 1.0)
print(f'\nFinal ensemble: range [{final.min():.4f}, {final.max():.4f}] mean {final.mean():.4f}')

In [ ]:
# ============ SUBMISSION ============
sub = pd.DataFrame({'filename': df_test['filename'], 'FaceOcclusion': final, 'gender': 'x'})
out = os.path.join(CFG['output_dir'], 'test_predictions.csv')
sub.to_csv(out, index=False)

# Kaggle needs submission in /kaggle/working
if IS_KAGGLE:
    sub.to_csv('/kaggle/working/test_predictions.csv', index=False)
else:
    sub.to_csv('test_predictions.csv', index=False)

print(f'Submission saved: {out}')
print(f'Shape: {sub.shape}')
print(f'\nPredictions distribution:')
print(sub['FaceOcclusion'].describe())
sub.head(10)

In [ ]:
# ============ FINAL SUMMARY ============
print('=' * 70)
print('FINAL RESULTS SUMMARY')
print('=' * 70)

for name, meta in model_meta.items():
    scores = [r['score'] for r in meta['results']]
    print(f'\n{name}:')
    for fold, r in zip(CFG['train_folds'], meta['results']):
        print(f'  Fold {fold}: {r["score"]:.6f}')
    print(f'  Mean CV: {np.mean(scores):.6f} +/- {np.std(scores):.6f}')

print(f'\nEnsemble weights: {dict((k[:30], f"{v:.3f}") for k,v in blend_w.items())}')
print(f'Blended OOF score: {s:.6f}')
print(f'\nTest predictions: [{final.min():.4f}, {final.max():.4f}] mean={final.mean():.4f}')
print(f'Submission: test_predictions.csv ({len(sub)} rows)')
print('=' * 70)

In [ ]:
# ============ VISUALIZE ============
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution comparison
axes[0].hist(df_train['FaceOcclusion'], bins=80, density=True, alpha=0.4, color='coral', label='Train GT')
axes[0].hist(final, bins=80, density=True, alpha=0.6, color='steelblue', label='Test pred')
axes[0].legend(); axes[0].set_title('Distribution: Train GT vs Test Predictions')

# Per-model OOF scores
model_names_short = [n.split('.')[0][:20] for n in model_meta.keys()]
oof_scores = []
for name in model_meta:
    mask = oof_filled[name]
    sc, _, _ = official_score(oof_pred[name][mask], y[mask], g[mask])
    oof_scores.append(sc)
axes[1].bar(model_names_short, oof_scores, color='steelblue', alpha=0.7)
axes[1].axhline(s, color='red', linestyle='--', label=f'Ensemble: {s:.6f}')
axes[1].set_title('OOF Score per Model'); axes[1].legend()
axes[1].set_ylabel('Score (lower=better)')

# Training curves (last model)
last_name = list(model_meta.keys())[-1]
for i, res in enumerate(model_meta[last_name]['results'][:3]):
    pass  # curves would need history stored
axes[2].hist(final, bins=100, color='steelblue', alpha=0.7)
axes[2].set_title(f'Test Predictions Distribution (n={len(final):,})')

plt.tight_layout(); plt.show()